# V8 arm isolation: TRACE rank-transition features (DAiL bnty_0716)

One V8 arm, shipped on its own: **TRACE rank-transition features** measured against the **V7 control**
with a **k=2 prompt-grouped OOF** check on public data.

* **Data** - `data/passages_clean.csv.gz`, the clean funnel subset of [prvsiyan/extended-name-that-llm-dataset](https://www.kaggle.com/datasets/prvsiyan/extended-name-that-llm-dataset) (CC BY 4.0, AI-generated text): 5,070 passages, 296 prompt blocks, 18 Name That LLM labels. Every passage in a block answered the *same* prompt, so topic is held fixed and only the author varies - which is both what a prompt-grouped check needs and what makes within-block ranks well defined. See `data/DATASET_CARD.md`.
* **Metric** - balanced accuracy over 18 classes (the house V-series metric), plus log-loss as the calibration gate.
* **Rule of this notebook** - the only difference between the two arms is the feature block; the classifier, its hyperparameters, the folds and the seeds are identical. So any delta is attributable to the arm, not to tuning.

Run it end to end: `Runtime > Run all`. No API keys, no downloads - the dataset ships in the repo.


## 1. Load the corpus and check the structure the folds depend on

Prompt blocks are the unit of validation: a random row split would put the same prompt in train and
validation and badly overstate accuracy, so every fold here moves whole blocks.

In [1]:
import json, os, platform, sys, time
os.environ["OMP_NUM_THREADS"] = os.environ["OPENBLAS_NUM_THREADS"] = os.environ["MKL_NUM_THREADS"] = "1"
sys.path.insert(0, os.getcwd())

import numpy as np, pandas as pd, sklearn
from ntl_v8.data import classes_of, load_corpus
from ntl_v8.features import VIEWS

print("python", platform.python_version(), "| numpy", np.__version__,
      "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

DATA = os.path.join("data", "passages_clean.csv.gz")
df = load_corpus(DATA)
classes = classes_of(df)
print(f"rows {len(df)}  prompt blocks {df['group'].nunique()}  labels {len(classes)}")
pd.DataFrame({"splits": df.groupby("split").size(),
              "blocks": df.groupby("split")["group"].nunique()}).sort_index()

python 3.12.8 | numpy 2.5.4 | pandas 3.1.1 | scikit-learn 1.9.1
rows 5070  prompt blocks 296  labels 18


,splits,blocks
split,,
new,4158,231
new2,144,8
proxy_v1,516,43
train_regen,252,14


## 2. The two arms

**V7 control** (documented re-implementation of the house baseline): TF-IDF word (1-2 grams, capped at
40k) + TF-IDF char\_wb (2-5 grams, capped at 40k) + a 14-column stylometry block, all fitted on the
training rows only, feeding one multinomial logistic regression (`C=4`, `class_weight='balanced'`, lbfgs, seed 0).

**V8-TRACE arm**: the identical control, plus 30 **rank-transition** columns. For each of the 14 style
views, take the document's rank *inside its own prompt block* (topic held fixed), then add the step-to-step
transitions between consecutive views plus three summaries of that transition sequence:

```
ranks   r_v(d) = rank of view v among d's block siblings, scaled to [0, 1]      (14 columns)
steps   t_v(d) = r_{v+1}(d) - r_v(d)                                            (13 columns)
summ    max |t|, mean |t|, number of sign flips in t                            ( 3 columns)
```

Why it should help: the absolute level of a style view is mostly topic and length protocol; its *position
inside the block* is what the author moves. And a model's register shows up as a *pattern of steps* across
views (e.g. high function-word rank with low punctuation rank), which a single-view feature cannot express.

**Leakage rules, enforced by tests:** TRACE uses no labels; a row's features depend only on its own group,
so computing them on the train side and on the validation side gives identical numbers; every transform is
fitted on training rows; no prompt block ever straddles a fold.

The view list, on real rows (raw values before any scaling):

In [2]:
from ntl_v8.features import style_block, trace_columns

views = pd.DataFrame([VIEWS], columns=[f"view {i+1}" for i in range(len(VIEWS))])
example = style_block(df["text"].head(3)).T.round(4)
example.index = VIEWS
print("TRACE feature columns:", len(trace_columns()))
display(views.T)
example

TRACE feature columns: 30


,0
view 1,n_words
view 2,mean_word_len
view 3,function_share
view 4,type_token_ratio
view 5,repeat_trigram_share
view 6,punct_per_100w
view 7,comma_per_100w
view 8,sent_per_100w
view 9,avg_sentence_words
view 10,digit_token_share


,0,1,2
n_words,82.0000,62.0000,71.0000
mean_word_len,6.0976,6.5323,5.9859
function_share,0.3171,0.2742,0.3239
type_token_ratio,0.9268,0.9355,0.9155
repeat_trigram_share,0.0000,0.0000,0.0000
punct_per_100w,24.3902,20.9677,22.5352
comma_per_100w,10.9756,11.2903,11.2676
sent_per_100w,4.8780,4.8387,5.6338
avg_sentence_words,20.5000,20.6667,17.7500
digit_token_share,0.0000,0.0000,0.0000


## 3. The gates

| gate | what it measures | pass rule |
|---|---|---|
| 0 | corpus shape: rows, blocks, labels, per-label spread | reported, not judged |
| 1 | fold integrity: no block shared between train and validation, every row scored exactly once | zero shared blocks |
| 2 | **primary k=2 prompt-grouped OOF delta** (arm - control) on all blocks | delta > 0, reported with per-fold numbers |
| 3 | stability across subsampled folds: 5 runs, each on a seeded 80% of the *blocks* | positive in >= 80% of runs |
| 4 | 400-draw bootstrap over prompt blocks (blocks resampled with replacement) | 95% CI above zero |
| 5 | damped calibration arm (`p' = 0.85 p + 0.15/K`) - the cheap second V8 arm, post-hoc only | reported; it cannot move balanced accuracy by construction |
| 6 | unscaled-TRACE variant - does standardising the rank block matter? | reported |

Everything is seeded and single-threaded, so a re-run reproduces the numbers bit for bit on the same library versions.

In [3]:
from ntl_v8.oof import run_gates

t0 = time.time()
report = run_gates(df, classes, reps=5, subsample=0.8, n_splits=2, seed=0, variants=True)
report["runtime_seconds"] = round(time.time() - t0, 1)
with open(os.path.join("evidence", "oof_report_notebook.json"), "w", encoding="utf-8") as fh:
    json.dump(report, fh, indent=2, sort_keys=True)
print("saved evidence/oof_report_notebook.json")

GATE k=2 prompt-grouped OOF delta (V8-TRACE - V7-control): +0.0227 balanced accuracy
  per-fold control ['0.2872', '0.2700']
  per-fold arm     ['0.3032', '0.2993']
GATE subsampled stability (5 runs @ 80% of prompt groups): mean +0.0133, min +0.0096, max +0.0173, positive 5/5
GATE prompt-group bootstrap 95% CI of delta: [+0.0120, +0.0326] (mean +0.0220, positive in 100.0% of draws)
GATE log-loss delta: -0.1038
VERDICT: PASS - the arm must beat the control on the k=2 OOF delta, with a bootstrap CI above zero and positive in >=80% of subsamples
saved evidence/oof_report_notebook.json


### Gate 2 headline numbers

In [4]:
g2, g3, g4 = report["gate2_primary_k2"], report["gate3_subsampled"], report["gate4_bootstrap"]
rows = [
    ("k=2 OOF balanced accuracy - V7 control", g2["control"]["balanced_accuracy"]),
    ("k=2 OOF balanced accuracy - V8-TRACE arm", g2["arm"]["balanced_accuracy"]),
    ("delta (the number the bounty asks for)", g2["delta_balanced_accuracy"]),
    ("delta macro-F1", g2["delta_macro_f1"]),
    ("delta log-loss (negative = better)", g2["delta_log_loss"]),
    ("bootstrap mean delta over prompt blocks", g4["mean_delta"]),
]
tbl = pd.DataFrame(rows, columns=["gate 2 / 4 number", "value"])
tbl["value"] = tbl["value"].map(lambda v: f"{v:+.4f}")
tbl

,gate 2 / 4 number,value
0,k=2 OOF balanced accuracy - V7 control,+0.2786
1,k=2 OOF balanced accuracy - V8-TRACE arm,+0.3013
2,delta (the number the bounty asks for),+0.0227
3,delta macro-F1,+0.0260
4,delta log-loss (negative = better),-0.1038
5,bootstrap mean delta over prompt blocks,+0.0220


### Gate 3 subsampled runs, gates 4-6

In [5]:
display(pd.DataFrame(report["gate3_subsampled"]["runs"])[
    ["seed", "groups_used", "control_ba", "arm_ba", "delta_balanced_accuracy"]
].round(4).assign(subsample=report["gate3_subsampled"]["subsample"]))
print("stability summary:", json.dumps(report["gate3_subsampled"]["summary"], indent=None))
print()
print("bootstrap  :", json.dumps(g4))
print()
print("damped calibration (gate 5):", json.dumps(report["gate5_damped_calibration"]))
print()
print("unscaled TRACE variant (gate 6):", json.dumps(report.get("gate6_trace_unscaled")))
print()
print("per-fold balanced accuracy - control", [round(v, 4) for v in g2["per_fold_control_ba"]],
      "| arm", [round(v, 4) for v in g2["per_fold_arm_ba"]])

,seed,groups_used,control_ba,arm_ba,delta_balanced_accuracy,subsample
0,1,237,0.2748,0.2859,0.0111,0.8
1,2,237,0.2774,0.2913,0.0140,0.8
2,3,237,0.2827,0.2922,0.0096,0.8
3,4,237,0.2709,0.2882,0.0173,0.8
4,5,237,0.2748,0.2893,0.0145,0.8


stability summary: {"n_runs": 5, "mean_delta": 0.013288689839386914, "min_delta": 0.009553948161543091, "max_delta": 0.0173118182720976, "std_delta": 0.003029926881946468, "positive_runs": 5}

bootstrap  : {"n_boot": 400, "mean_delta": 0.021963602856727535, "ci95_low": 0.012005850161650882, "ci95_high": 0.03258918433644414, "share_positive": 1.0}

damped calibration (gate 5): {"alpha": 0.15, "log_loss_before": 2.2295631419572914, "log_loss_after": 2.2160910051884373, "delta_log_loss": -0.01347213676885417, "delta_balanced_accuracy": 0.0}

unscaled TRACE variant (gate 6): {"balanced_accuracy": 0.30721803819629906, "log_loss": 2.1175545165987226, "delta_balanced_vs_control": 0.028622001448088408, "per_fold_ba": [0.31202631202631204, 0.3024136341065475], "seconds": 474.33}

per-fold balanced accuracy - control [0.2872, 0.27] | arm [0.3032, 0.2993]


## 4. Reading the result honestly

What the numbers above say, stated plainly and without spin:

* The delta is measured on **this public corpus**, not on the Kaggle leaderboard: the dataset host's own
  prompts are unpublished, so `train_regen` rows use community-reconstructed prompt texts. Absolute accuracy
  here (~0.29 over 18 classes) reflects how hard sibling attribution is when the topic is held fixed - it is a
  signal-isolation harness, not a submission attempt.
* The gate that matters for the bounty is the **delta**, and it must survive fold subsampling (gate 3) and the
  block bootstrap (gate 4). A single lucky k=2 split is not evidence.
* TRACE is one arm. This notebook deliberately does *not* stack arms, so whatever delta appears belongs to
  rank-transition features alone - which is the isolation the house asked for.
* Falsification: re-running with different seeds (`--seed`) or a different `--subsample` share, or swapping the
  classifier (e.g. linear SVM), should keep the sign of the delta. If it flips, the signal is fold noise and
  the arm should be dropped - the harness prints FAIL in that case rather than hiding it.

Reproduce from the shell instead of the notebook:

```bash
python run_oof.py --reps 5 --boot 400 --out evidence/oof_report.json   # ~17-27 min, one core
python -m pytest tests -q                                              # ~30 s, 19 checks
```